# 00 - Generate synthetic EHR extract for Lakeview Health

Creates a **fully synthetic** EHR extract for a fictional 3-hospital system (no real PHI) and lands it as files in a Unity Catalog volume. A Lakeflow pipeline then ingests the files with Auto Loader.

| Entity | Format | Notes |
| --- | --- | --- |
| patients | CSV | Demographics; name, DOB and MRN are PHI |
| encounters | CSV | Inpatient stays from 2025-01 to 2026-11-03 |
| diagnoses, medications, labs | CSV | Clinical detail per encounter |
| discharge_notes | JSON | Free-text discharge summaries that contain **social risk signals** |
| followup_appointments | CSV | Post-discharge appointments |
| care_mgmt_outreach | CSV | Legacy outreach log (coordinators pick patients with LACE >= 10) |

**Built-in ground truth:** 30-day readmission risk depends on clinical factors and on social factors (lives alone, transport, medication cost, health literacy, substance use) that appear **only in the note text**. That is why the GenAI-extracted features should improve the model over LACE.

In [0]:
import numpy as np, pandas as pd, json, os
from types import SimpleNamespace

CATALOG = "simo_show_case_catalog"
LANDING = f"/Volumes/{CATALOG}/lakeview_raw/landing"
N_PATIENTS = 8000          # synthetic sample (about 20% of Lakeview's annual volume)
SEED = 42
B0 = -3.45                 # logit intercept, calibrated for a 30-day readmission rate of about 16%
START = pd.Timestamp("2025-01-01")
CUTOFF = pd.Timestamp("2026-11-03 23:59:00")   # extract date: "yesterday" relative to the demo
TODAY = pd.Timestamp("2026-11-04")
rng = np.random.default_rng(SEED)

In [0]:
FIRST_F = ["Mary","Linda","Patricia","Barbara","Susan","Maria","Rosa","Thi","Karen","Nancy","Dorothy","Helen","Ana","Grace","Ruth"]
FIRST_M = ["James","John","Robert","Michael","William","David","Jose","Luis","Minh","Charles","Thomas","Richard","Joseph","George","Frank"]
LAST = ["Smith","Johnson","Williams","Brown","Jones","Garcia","Miller","Davis","Rodriguez","Martinez","Nguyen","Lopez","Wilson","Anderson","Taylor","Thomas","Moore","Jackson","Lee","Harris"]
FAC = {"LVH-NORTH":["60601","60602","60610"], "LVH-CENTRAL":["60614","60615","60616"], "LVH-SOUTH":["60619","60620","60628"]}
FAC_NAMES, FAC_P = list(FAC), [0.35, 0.40, 0.25]

pts = []
for i in range(N_PATIENTS):
    senior = rng.random() < 0.62
    age = int(np.clip(rng.normal(75, 8), 65, 98)) if senior else int(rng.integers(25, 65))
    sex = "F" if rng.random() < 0.53 else "M"
    payer = (rng.choice(["Medicare","Medicare Advantage","Commercial"], p=[0.6,0.3,0.1]) if age >= 65
             else rng.choice(["Medicaid","Commercial","Self-pay"], p=[0.35,0.55,0.10]))
    lang = rng.choice(["English","Spanish","Vietnamese","Polish"], p=[0.84,0.11,0.03,0.02])
    fac = rng.choice(FAC_NAMES, p=FAC_P)
    low_income = payer in ("Medicaid", "Self-pay")
    lives_alone = rng.random() < 0.14 + 0.16*(age >= 75)
    pts.append(dict(
        patient_id=f"P{i+1:06d}", mrn=f"MRN{rng.integers(10**7, 10**8)}",
        first_name=rng.choice(FIRST_F if sex == "F" else FIRST_M), last_name=rng.choice(LAST),
        dob=(TODAY - pd.Timedelta(days=int(age*365.25 + rng.integers(0, 365)))).date().isoformat(),
        sex=sex, zip=rng.choice(FAC[fac]), payer=payer, primary_language=lang, home_facility_id=fac,
        # hidden ground truth (NOT written as structured columns, only leaks through note text)
        h_age=age, h_lives_alone=lives_alone, h_caregiver=(not lives_alone) and rng.random() < 0.7,
        h_transport=rng.random() < 0.07 + 0.10*low_income + 0.04*(age >= 80),
        h_med_cost=rng.random() < 0.05 + 0.14*low_income,
        h_literacy=rng.random() < 0.06 + 0.12*(lang != "English"),
        h_substance=rng.random() < 0.05 + 0.07*(age < 65),
        h_charlson=int(min(10, rng.poisson(0.6 + 0.045*max(age-45, 0)))),
        h_frailty=rng.normal(0, 0.45)))
pts_df = pd.DataFrame(pts)
print(f"{len(pts_df):,} patients")
print(pts_df.payer.value_counts(normalize=True).round(3).to_dict())
print({c: round(pts_df[c].mean(), 3) for c in ["h_lives_alone","h_transport","h_med_cost","h_literacy","h_substance"]})

8,000 patients
{np.str_('Medicare'): 0.367, np.str_('Commercial'): 0.268, np.str_('Medicare Advantage'): 0.185, np.str_('Medicaid'): 0.141, np.str_('Self-pay'): 0.038}
{'h_lives_alone': np.float64(0.19), 'h_transport': np.float64(0.092), 'h_med_cost': np.float64(0.071), 'h_literacy': np.float64(0.08), 'h_substance': np.float64(0.078)}


In [0]:
COND = {
 "Heart failure":               dict(icd="I50.23",  p=0.15, los=(2.2,2.2), eff=0.45),
 "COPD exacerbation":           dict(icd="J44.1",   p=0.11, los=(2.0,2.0), eff=0.35),
 "Pneumonia":                   dict(icd="J18.9",   p=0.13, los=(2.0,2.2), eff=0.20),
 "Acute MI":                    dict(icd="I21.4",   p=0.06, los=(2.0,2.0), eff=0.15),
 "Sepsis":                      dict(icd="A41.9",   p=0.09, los=(2.5,2.4), eff=0.30),
 "Hip/knee replacement":        dict(icd="M17.11",  p=0.07, los=(2.0,1.2), eff=-0.90),
 "Diabetes with complications": dict(icd="E11.65",  p=0.07, los=(2.0,1.8), eff=0.25),
 "Cellulitis":                  dict(icd="L03.115", p=0.08, los=(2.0,1.6), eff=-0.10),
 "GI bleed":                    dict(icd="K92.2",   p=0.07, los=(2.0,1.7), eff=0.20),
 "Urinary tract infection":     dict(icd="N39.0",   p=0.09, los=(2.0,1.5), eff=0.05),
 "Acute kidney injury":         dict(icd="N17.9",   p=0.08, los=(2.0,2.0), eff=0.30),
}
CNAMES = list(COND); CP = np.array([COND[c]["p"] for c in CNAMES]); CP = CP / CP.sum()

def lace_score(los, atype, charlson, ed6):
    L = 0 if los < 1 else 1 if los == 1 else 2 if los == 2 else 3 if los == 3 else 4 if los <= 6 else 5 if los <= 13 else 7
    A = 3 if atype in ("Emergency", "Urgent") else 0
    C = charlson if charlson < 4 else 5
    return L + A + C + min(ed6, 4)

enc, eid = [], 0
for r in (SimpleNamespace(**d) for d in pts_df.to_dict("records")):
    spont = 1 + min(4, rng.poisson(0.35*(1 + r.h_charlson/3)))
    t = START + pd.Timedelta(days=float(rng.uniform(0, 600)))
    forced_cond, atype, planned = None, None, False
    while t < CUTOFF:
        cond = forced_cond or rng.choice(CNAMES, p=CP)
        if cond == "Hip/knee replacement" and r.h_age < 50 and forced_cond is None:
            cond = "Cellulitis"
        if atype is None:
            atype = "Elective" if cond == "Hip/knee replacement" else rng.choice(["Emergency","Urgent"], p=[0.8,0.2])
        sh, sc = COND[cond]["los"]
        los = max(1, int(round(rng.gamma(sh, sc) + 0.3*r.h_charlson)))
        admit = t.replace(hour=int(rng.integers(0,24)), minute=int(rng.integers(0,60)), second=0, microsecond=0)
        disch = admit.normalize() + pd.Timedelta(days=los, hours=int(rng.integers(10,19)), minutes=int(rng.integers(0,60)))
        if disch > CUTOFF:
            break
        age_at = r.h_age - (TODAY - admit).days/365.25
        u = rng.random()
        p_exp = 0.008 + 0.004*r.h_charlson + (0.02 if cond == "Sepsis" else 0)
        p_ama = 0.015 + 0.06*r.h_substance
        p_snf = 0.04 + 0.12*(age_at >= 75) + (0.25 if cond == "Hip/knee replacement" else 0)
        dispo = ("Expired" if u < p_exp else "Left against medical advice" if u < p_exp+p_ama
                 else "Skilled nursing facility" if u < p_exp+p_ama+p_snf else "Home with home health" if u < p_exp+p_ama+p_snf+0.18 else "Home")
        ed6 = int(rng.poisson(0.35 + 0.25*r.h_charlson + 0.6*r.h_substance + 0.3*r.h_transport))
        nmeds = int(rng.poisson(4 + 1.4*r.h_charlson + (2 if cond in ("Heart failure","Acute MI") else 0)))
        sodium = rng.normal(136 if cond == "Heart failure" else 139, 3.5)
        hgb = rng.normal(10.5 if cond == "GI bleed" else 12.6, 1.7)
        creat = float(np.exp(rng.normal(0.1 + (0.6 if cond == "Acute kidney injury" else 0) + 0.05*r.h_charlson, 0.35)))
        bnp = float(np.exp(rng.normal(6.6 if cond == "Heart failure" else 4.5, 0.7)))
        fu_sched = rng.random() < 0.62 - 0.15*r.h_transport - 0.10*r.h_literacy
        fu_attend = fu_sched and (rng.random() < 0.82 - 0.35*r.h_transport - 0.15*r.h_med_cost)
        lace = lace_score(los, atype, r.h_charlson, ed6)
        hist = disch < pd.Timestamp("2026-10-27")   # legacy outreach program only for older discharges
        contacted = hist and dispo != "Expired" and (rng.random() < (0.40 if lace >= 10 else 0.06))
        reached = contacted and rng.random() < 0.68 - 0.2*r.h_literacy - 0.1*r.h_transport
        logit = (B0 + 0.20*r.h_charlson + 0.05*min(los,15) + 0.28*min(ed6,4) + 0.40*(atype != "Elective")
                 + COND[cond]["eff"] + 0.60*r.h_lives_alone + 0.55*r.h_transport + 0.70*r.h_med_cost + 0.30*r.h_literacy
                 + 0.45*r.h_substance + 0.20*(age_at >= 75) + 0.90*(dispo == "Left against medical advice") + 0.15*(dispo == "Skilled nursing facility")
                 + 0.25*(sodium < 133) + 0.25*(hgb < 10) + 0.20*(creat > 1.6) + 0.35*(bnp > 800) + 0.25*(nmeds >= 10)
                 - 0.55*fu_attend - 0.50*reached - 0.35*r.h_caregiver + r.h_frailty)
        p = 1/(1+np.exp(-logit))
        readmit = dispo != "Expired" and rng.random() < p
        eid += 1
        enc.append(dict(encounter_id=f"E{eid:07d}", patient_id=r.patient_id,
            facility_id=r.home_facility_id if rng.random() < 0.9 else rng.choice(FAC_NAMES),
            encounter_type="Inpatient", admission_type=atype, is_planned=planned,
            admit_ts=admit, discharge_ts=disch, primary_condition=cond, primary_icd10=COND[cond]["icd"],
            discharge_disposition=dispo, ed_visits_prior_6m=ed6,
            h_r=r, h_los=los, h_nmeds=nmeds, h_sodium=round(sodium,1), h_hgb=round(hgb,1), h_creat=round(creat,2), h_bnp=int(bnp),
            h_fu_sched=fu_sched, h_fu_attend=fu_attend, h_lace=lace, h_contacted=contacted, h_reached=reached, h_p=p, h_readmit=readmit))
        forced_cond, atype, planned = None, None, False
        if dispo == "Expired":
            break
        if readmit:
            t = disch + pd.Timedelta(days=float(rng.uniform(1, 29.5)))
            forced_cond = cond if rng.random() < 0.5 else None
            atype = "Emergency"
        elif rng.random() < 0.03:   # planned staged procedure: excluded from the readmission measure
            t = disch + pd.Timedelta(days=float(rng.uniform(5, 28)))
            forced_cond, atype, planned = rng.choice(["Hip/knee replacement","Acute MI"]), "Elective", True
        else:
            spont -= 1
            if spont <= 0:
                break
            t = disch + pd.Timedelta(days=float(31 + rng.exponential(160)))

enc_df = pd.DataFrame(enc)
mature = enc_df[enc_df.discharge_ts < CUTOFF - pd.Timedelta(days=30)]
print(f"{len(enc_df):,} encounters | simulated 30-day readmission rate (discharges with a full 30-day window): {mature.h_readmit.mean():.1%}")
print(f"Discharges per day, last 7 days: {(enc_df.discharge_ts >= CUTOFF - pd.Timedelta(days=7)).sum()/7:.1f}")
print("LACE >= 10 share:", round((enc_df.h_lace >= 10).mean(), 3), "| contacted:", round(enc_df.h_contacted.mean(), 3), "| reached:", round(enc_df.h_reached.mean(), 3))

13,200 encounters | simulated 30-day readmission rate (discharges with a full 30-day window): 15.5%
Discharges per day, last 7 days: 7.7
LACE >= 10 share: 0.381 | contacted: 0.184 | reached: 0.119


In [0]:
COMORB = [("I10","Essential hypertension"),("E11.9","Type 2 diabetes"),("N18.3","CKD stage 3"),("I48.91","Atrial fibrillation"),
          ("F32.9","Major depressive disorder"),("E78.5","Hyperlipidemia"),("J44.9","COPD"),("I25.10","Coronary artery disease"),
          ("F03.90","Dementia"),("E66.9","Obesity"),("D64.9","Anemia"),("I50.9","Heart failure, unspecified")]
MEDS = {"Heart failure":["furosemide","carvedilol","sacubitril/valsartan","spironolactone"],"COPD exacerbation":["prednisone","tiotropium inhaler","albuterol inhaler","azithromycin"],
        "Pneumonia":["amoxicillin/clavulanate","azithromycin"],"Acute MI":["aspirin","ticagrelor","atorvastatin","metoprolol"],"Sepsis":["cefuroxime","acetaminophen"],
        "Hip/knee replacement":["apixaban","oxycodone","acetaminophen"],"Diabetes with complications":["insulin glargine","insulin lispro","metformin"],
        "Cellulitis":["cephalexin"],"GI bleed":["pantoprazole","ferrous sulfate"],"Urinary tract infection":["nitrofurantoin"],"Acute kidney injury":["sodium bicarbonate"]}
CHRONIC_MEDS = ["lisinopril","amlodipine","metformin","atorvastatin","levothyroxine","sertraline","omeprazole","gabapentin","warfarin","tamsulosin","donepezil","insulin glargine"]
SPECIALTY = {"Heart failure":"Cardiology","Acute MI":"Cardiology","COPD exacerbation":"Pulmonology","Hip/knee replacement":"Orthopedics","Diabetes with complications":"Endocrinology","Acute kidney injury":"Nephrology"}
COORD = {"LVH-NORTH":["CC01","CC02","CC03","CC04"],"LVH-CENTRAL":["CC05","CC06","CC07","CC08","CC09"],"LVH-SOUTH":["CC10","CC11","CC12"]}

dx, meds, labs, fups, outreach = [], [], [], [], []
for e in enc_df.itertuples(index=False):
    r = e.h_r
    dx.append(dict(encounter_id=e.encounter_id, seq=1, icd10_code=e.primary_icd10, description=e.primary_condition, is_primary=True))
    for k, (code, desc) in enumerate([COMORB[j] for j in rng.choice(len(COMORB), size=min(len(COMORB), r.h_charlson + int(rng.integers(0,2))), replace=False)]):
        dx.append(dict(encounter_id=e.encounter_id, seq=k+2, icd10_code=code, description=desc, is_primary=False))
    med_list = list(MEDS[e.primary_condition]) + list(rng.choice(CHRONIC_MEDS, size=min(len(CHRONIC_MEDS), max(0, e.h_nmeds - len(MEDS[e.primary_condition]))), replace=False))
    for m in med_list[:max(1, e.h_nmeds)]:
        meds.append(dict(encounter_id=e.encounter_id, med_name=m, prescribed_on_discharge=True))
    rts = e.discharge_ts - pd.Timedelta(hours=8)
    for code, name, val, unit in [("2951-2","Sodium",e.h_sodium,"mmol/L"),("718-7","Hemoglobin",e.h_hgb,"g/dL"),("2160-0","Creatinine",e.h_creat,"mg/dL"),("30934-4","BNP",e.h_bnp,"pg/mL")]:
        labs.append(dict(encounter_id=e.encounter_id, loinc_code=code, lab_name=name, value=val, unit=unit, result_ts=rts))
    if e.h_fu_sched and e.discharge_disposition != "Expired":
        appt = e.discharge_ts.normalize() + pd.Timedelta(days=int(rng.integers(3, 8)), hours=int(rng.integers(8, 17)))
        status = "Scheduled" if appt > CUTOFF else ("Completed" if e.h_fu_attend else "No-show")
        fups.append(dict(appointment_id=f"A{e.encounter_id[1:]}", patient_id=e.patient_id, source_encounter_id=e.encounter_id, appt_ts=appt,
                         provider_type=SPECIALTY.get(e.primary_condition, "Primary care"), status=status))
    if e.h_contacted:
        outreach.append(dict(outreach_id=f"O{e.encounter_id[1:]}", encounter_id=e.encounter_id, patient_id=e.patient_id,
                             coordinator_id=rng.choice(COORD[e.facility_id]), outreach_ts=e.discharge_ts + pd.Timedelta(hours=float(rng.uniform(16, 70))),
                             channel="Phone", outcome="Reached" if e.h_reached else rng.choice(["No answer","Left voicemail","Wrong number"], p=[0.5,0.4,0.1])))
dx_df, meds_df, labs_df, fups_df, outreach_df = map(pd.DataFrame, [dx, meds, labs, fups, outreach])
coord_df = pd.DataFrame([dict(coordinator_id=c, facility_id=f) for f, cs in COORD.items() for c in cs])
print({k: len(v) for k, v in dict(diagnoses=dx_df, medications=meds_df, labs=labs_df, followup_appointments=fups_df, care_mgmt_outreach=outreach_df, coordinators=coord_df).items()})

{'diagnoses': 42783, 'medications': 90222, 'labs': 52800, 'followup_appointments': 7816, 'care_mgmt_outreach': 2433, 'coordinators': 12}


In [0]:
PHRASES = {
 "lives_alone": ["Patient lives alone in a second-floor apartment.", "Widowed and lives by {pro_ref}; no one at home to assist.", "Reports living alone since spouse passed away last year."],
 "caregiver":   ["{Child} will stay with the patient for the first week after discharge.", "Spouse at home and able to help with medications.", "Lives with family who are involved in care."],
 "transport":   ["Patient does not drive and has no reliable transportation to follow-up appointments.", "Concerned about getting to clinic visits; relies on the bus and has missed appointments before.", "No car; neighbor occasionally provides rides."],
 "med_cost":    ["States {pro} cannot afford the copay for {med}.", "Reports skipping doses of {med} to make prescriptions last because of cost.", "Has been rationing {med} due to financial strain."],
 "literacy":    ["Teach-back incomplete; patient had difficulty restating discharge instructions.", "Limited health literacy noted; {lang} interpreter used for discharge teaching.", "Patient unsure how to use new medications despite education."],
 "substance":   ["Current smoker, about one pack per day.", "Reports daily alcohol use, 4-6 drinks per day.", "History of opioid use disorder, not currently in treatment."],
 "neutral":     ["Lives with spouse in a single-family home.", "Independent with activities of daily living.", "Retired; enjoys gardening.", "Denies tobacco, alcohol, or drug use."],
}
COURSE = {"Heart failure":"was diuresed with IV furosemide with improvement in dyspnea and edema","COPD exacerbation":"was treated with nebulized bronchodilators, steroids and antibiotics",
          "Pneumonia":"was treated with IV antibiotics and transitioned to oral therapy","Acute MI":"underwent cardiac catheterization with PCI to the LAD",
          "Sepsis":"was resuscitated with IV fluids and broad-spectrum antibiotics, source urinary","Hip/knee replacement":"underwent an uncomplicated joint replacement and worked with physical therapy",
          "Diabetes with complications":"was managed with an insulin drip, then transitioned to a basal-bolus regimen","Cellulitis":"was treated with IV cefazolin with resolution of erythema",
          "GI bleed":"received 2 units of PRBCs and underwent EGD with clipping of a gastric ulcer","Urinary tract infection":"was treated with IV ceftriaxone","Acute kidney injury":"was managed with IV fluids and holding nephrotoxic medications"}

def pick(key, **kw):
    return rng.choice(PHRASES[key]).format(**kw)

notes = []
for e in enc_df.itertuples(index=False):
    r = e.h_r
    age_at = int(r.h_age - (TODAY - e.admit_ts).days/365.25)
    pro, pro_ref = ("she", "herself") if r.sex == "F" else ("he", "himself")
    kw = dict(pro=pro, pro_ref=pro_ref, Child=rng.choice(["Daughter","Son"]), med=rng.choice(MEDS[e.primary_condition]), lang=r.primary_language)
    social = []
    for key, flag in [("lives_alone", r.h_lives_alone), ("caregiver", r.h_caregiver), ("transport", r.h_transport), ("med_cost", r.h_med_cost),
                      ("literacy", r.h_literacy), ("substance", r.h_substance)]:
        if (flag and rng.random() < 0.85) or (not flag and rng.random() < 0.02):   # imperfect documentation
            social.append(pick(key, **kw))
    if len(social) < 2:
        social.append(pick("neutral", **kw))
    rng.shuffle(social)
    if e.discharge_disposition == "Expired":
        plan = "Patient expired during hospitalization; family notified."
    else:
        fu = (f"Follow-up with {SPECIALTY.get(e.primary_condition, 'primary care')} scheduled within 7 days." if e.h_fu_sched
              else "No follow-up appointment scheduled at time of discharge; patient to call clinic.")
        plan = f"Discharged to {e.discharge_disposition.lower()}. {fu}"
    text = (f"DISCHARGE SUMMARY\nAdmission: {e.admit_ts:%Y-%m-%d} ({e.admission_type.lower()}). Discharge: {e.discharge_ts:%Y-%m-%d}.\n"
            f"HPI/Hospital course: {age_at}-year-old {'woman' if r.sex == 'F' else 'man'} admitted for {e.primary_condition.lower()}; {pro} {COURSE[e.primary_condition]}. "
            f"Length of stay {e.h_los} days.\nSocial history: {' '.join(social)}\n"
            f"Discharge medications: {e.h_nmeds} medications reconciled.\nPlan: {plan}")
    notes.append(dict(note_id=f"N{e.encounter_id[1:]}", encounter_id=e.encounter_id, patient_id=e.patient_id, note_type="Discharge summary",
                      author_role=rng.choice(["Hospitalist","Nurse practitioner","Physician assistant"]), note_ts=e.discharge_ts.isoformat(), note_text=text))
notes_df = pd.DataFrame(notes)
print(f"{len(notes_df):,} notes. Example:\n")
print(notes_df.note_text.iloc[3])

13,200 notes. Example:

DISCHARGE SUMMARY
Admission: 2025-12-29 (urgent). Discharge: 2026-01-03.
HPI/Hospital course: 67-year-old woman admitted for sepsis; she was resuscitated with IV fluids and broad-spectrum antibiotics, source urinary. Length of stay 5 days.
Social history: Patient unsure how to use new medications despite education. Independent with activities of daily living.
Discharge medications: 5 medications reconciled.
Plan: Discharged to home with home health. Follow-up with primary care scheduled within 7 days.


In [0]:
import shutil
public_cols = lambda df: [c for c in df.columns if not c.startswith("h_")]   # never write hidden ground truth
files = {
    "patients": pts_df[public_cols(pts_df)],
    "encounters": enc_df[public_cols(enc_df)],
    "diagnoses": dx_df, "medications": meds_df, "labs": labs_df,
    "followup_appointments": fups_df, "care_mgmt_outreach": outreach_df, "coordinators": coord_df,
}
for name, df in files.items():
    d = f"{LANDING}/{name}"
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d, exist_ok=True)
    df.to_csv(f"{d}/{name}_extract_20261103.csv", index=False)
d = f"{LANDING}/discharge_notes"
shutil.rmtree(d, ignore_errors=True); os.makedirs(d, exist_ok=True)
notes_df.to_json(f"{d}/discharge_notes_extract_20261103.json", orient="records", lines=True)
for name in list(files) + ["discharge_notes"]:
    print(f"{name:<24} {os.listdir(f'{LANDING}/{name}')}")

patients                 ['patients_extract_20261103.csv']
encounters               ['encounters_extract_20261103.csv']
diagnoses                ['diagnoses_extract_20261103.csv']
medications              ['medications_extract_20261103.csv']
labs                     ['labs_extract_20261103.csv']
followup_appointments    ['followup_appointments_extract_20261103.csv']
care_mgmt_outreach       ['care_mgmt_outreach_extract_20261103.csv']
coordinators             ['coordinators_extract_20261103.csv']
discharge_notes          ['discharge_notes_extract_20261103.json']


In [0]:
%sql
-- Read the landed files back with read_files (a different reader than pandas) to confirm row counts and parsing
SELECT 'patients' AS entity, count(*) AS rows FROM read_files('/Volumes/simo_show_case_catalog/lakeview_raw/landing/patients', format => 'csv', header => true)
UNION ALL SELECT 'encounters', count(*) FROM read_files('/Volumes/simo_show_case_catalog/lakeview_raw/landing/encounters', format => 'csv', header => true)
UNION ALL SELECT 'diagnoses', count(*) FROM read_files('/Volumes/simo_show_case_catalog/lakeview_raw/landing/diagnoses', format => 'csv', header => true)
UNION ALL SELECT 'medications', count(*) FROM read_files('/Volumes/simo_show_case_catalog/lakeview_raw/landing/medications', format => 'csv', header => true)
UNION ALL SELECT 'labs', count(*) FROM read_files('/Volumes/simo_show_case_catalog/lakeview_raw/landing/labs', format => 'csv', header => true)
UNION ALL SELECT 'followup_appointments', count(*) FROM read_files('/Volumes/simo_show_case_catalog/lakeview_raw/landing/followup_appointments', format => 'csv', header => true)
UNION ALL SELECT 'care_mgmt_outreach', count(*) FROM read_files('/Volumes/simo_show_case_catalog/lakeview_raw/landing/care_mgmt_outreach', format => 'csv', header => true)
UNION ALL SELECT 'discharge_notes', count(*) FROM read_files('/Volumes/simo_show_case_catalog/lakeview_raw/landing/discharge_notes', format => 'json')

entity,rows
patients,8000
encounters,13200
diagnoses,42783
medications,90222
labs,52800
followup_appointments,7816
care_mgmt_outreach,2433
discharge_notes,13200
